# 06a — The Stim copy of Selene's circuit

**Result.** A gate-for-gate copy of Selene's compiled surface-code memory, built in Stim from
Selene's own instruction log, is the same circuit with the same noise, to the precision these
checks reach:

| check | result |
| --- | --- |
| native gates | 24 `Rzz` for 24 CNOTs; single-qubit gates per data qubit = `n_ctrl + 3·n_tgt + Δt` on all 9 |
| gate conversion | Selene's 5-gate CNOT converts to exactly `CX`; a non-Clifford angle is refused |
| noise placement (Selene's log) | one Pauli **before** every native single-qubit gate (5 of 5); none on `Rzz`; `p_meas` flips the reported bit only |
| noiseless | zero detection events in the copy and in Selene, at Δt = 1, N = 3 and Δt = 8, N = 12 |
| noisy statistics | per-detector χ²/dof 1.07–1.33, pairwise 0.98–1.14 against Selene-vs-Selene and copy-vs-copy references of 0.90–1.55 and 1.05–1.08 |
| the same checks at d = 5 (§6b) | noiseless exactly zero; per-detector 0.89 / 1.07 against references 1.10 / 0.89; pairwise 0.98 / 0.99 against 1.01 / 0.99, over 12,074 pairs |

The copy is therefore a valid source of Selene's detector error model, which 06b uses to decode
Selene's shots. That model contains fault mechanisms a hand-built phenomenological graph does
not: diagonal ones (a fault part-way through extraction) carry 8.4% of all fault probability at
Δt = 1, falling to 2.7% at Δt = 32 as idle faults take over (d = 3), and 14.0% falling to 7.4%
over Δt = 1 to 16 at d = 5.

## Why a copy

A matching decoder needs a **detector error model**: every way a fault can happen, which
detectors each lights up, and how likely it is. Selene cannot provide one: it runs the circuit
and returns measurement results, and its noise model is a compiled library. Stim can compute
the complete detector error model of any circuit it is given, by pushing every possible fault
through every gate.

So Selene's circuit is copied into Stim, gate for gate, with Selene's noise in the same places
(`qec_timing.circuit_level.selene_mirror`). Stim then supplies the decoder's knowledge of the
circuit; **the data stays Selene's** (06b and 06c decode Selene's own shots). A decoder built
from the wrong circuit would be wrong in a new way, so this notebook checks the copy before
anything relies on it:

1. which native gates Selene actually runs (§1);
2. that each one is converted to the right Stim gate (§2);
3. where Selene puts its noise, read off its own instruction log (§3);
4. that the noiseless copy is deterministic (§4);
5. that the noisy copy's detector statistics match Selene's, measured against how much two
   independent blocks of Selene shots differ from each other (§5).

§6 then shows what the copy's error model contains that a hand-built decoder lacks.

In [1]:
import sys, pathlib, time, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '../src')
from qec_timing.circuit_level.surface_memory import Geometry
from qec_timing.circuit_level import selene_mirror as sm, memory_shots, runcache

pd.set_option('display.width', 160)
BLUE, ORANGE, AQUA, GRAY, INK, MUTED = '#2a78d6', '#eb6834', '#1baf7a', '#8a8a86', '#0b0b0b', '#52514e'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#c9c9c4', 'axes.labelcolor': MUTED, 'axes.titlesize': 10,
                     'axes.labelsize': 9, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'xtick.labelsize': 8, 'ytick.labelsize': 8, 'grid.color': '#e6e6e1',
                     'legend.frameon': False, 'legend.fontsize': 8, 'font.size': 9, 'text.color': INK})
FIGDIR = pathlib.Path('figures'); FIGDIR.mkdir(exist_ok=True)


def finish(fig, name):
    fig.tight_layout()
    fig.savefig(FIGDIR / f'{name}.png', bbox_inches='tight', facecolor='white')
    plt.show()

import math, tempfile, importlib.util, importlib.metadata as md
import stim
from selene_sim import build, Stim, IdealErrorModel, DepolarizingErrorModel
from selene_sim.event_hooks.instruction_log import CircuitExtractor

print({d: md.version(d) for d in ('selene-sim', 'selene-hugr-qis-compiler', 'guppylang',
                                  'stim', 'pymatching')})
CODE, DISTANCE = 'rotated', 3
G = Geometry(CODE, DISTANCE)
print(f"{CODE} d={DISTANCE}: {G.n_data} data qubits, {G.n_anc} ancillas "
      f"({len(G.zcols)} Z-checks decoded)")

{'selene-sim': '0.3.0', 'selene-hugr-qis-compiler': '0.4.2', 'guppylang': '1.0.3', 'stim': '1.16.0', 'pymatching': '2.4.0'}
rotated d=3: 9 data qubits, 8 ancillas (4 Z-checks decoded)


## 1. The native gates Selene runs

One noiseless shot of the memory program (Δt = 1 idle op per data qubit, one round), read back
through `CircuitExtractor.get_error_model_output()`: the exact operations the error model
passes to the simulator. Every CNOT compiles to one `Rzz` plus single-qubit `Rxy`/`Rz` gates,
and the idle `phased_x(q, 0, 0)` arrives as `Rxy(0, 0)`.

The second table counts single-qubit gates per data qubit against how many of its CNOTs it is
the control of and the target of. §2 shows the decomposition puts 1 gate on the control and
3 on the target, so the expectation is `n_ctrl + 3·n_tgt + Δt`.

In [2]:
TAU, R = 1, 1
ops = sm.native_ops(G._runner(TAU, R), G.n_data + G.n_anc)
print("operations in one round:", dict(collections.Counter(o['op'] for o in ops)))
print(f"CNOTs in the layout: {len(G.L['cx'])}")

data_ids = [o['qubit'] for o in ops if o['op'] == 'Reset'][:G.n_data]
one_q = collections.Counter(o['qubit'] for o in ops if o['op'] in ('Rxy', 'Rz'))
idle = collections.Counter(o['qubit'] for o in ops
                           if o['op'] == 'Rxy' and o['theta'] == 0.0 and o['phi'] == 0.0)
didx = {q: i for i, q in enumerate(G.L['data'])}
n_ctrl = np.zeros(G.n_data, int); n_tgt = np.zeros(G.n_data, int)
for c_, t_ in G.L['cx']:
    if c_ in didx: n_ctrl[didx[c_]] += 1
    if t_ in didx: n_tgt[didx[t_]] += 1
per_q = pd.DataFrame({'n_ctrl': n_ctrl, 'n_tgt': n_tgt,
                      'idle Rxy(0,0)': [idle[q] for q in data_ids],
                      '1q gates': [one_q[q] for q in data_ids],
                      'n_ctrl + 3 n_tgt + tau': n_ctrl + 3 * n_tgt + TAU},
                     index=[f'q{i}' for i in range(G.n_data)])
print("\nper data qubit, one round:\n")
print(per_q.to_string())
print("\nmatches n_ctrl + 3 n_tgt + tau on every data qubit:",
      bool((per_q['1q gates'] == per_q['n_ctrl + 3 n_tgt + tau']).all()))

operations in one round: {'Reset': 17, 'Rxy': 65, 'Rz': 56, 'Rzz': 24, 'MeasureRequest': 17}
CNOTs in the layout: 24

per data qubit, one round:

    n_ctrl  n_tgt  idle Rxy(0,0)  1q gates  n_ctrl + 3 n_tgt + tau
q0       1      1              1         5                       5
q1       1      2              1         8                       8
q2       1      1              1         5                       5
q3       2      1              1         6                       6
q4       2      2              1         9                       9
q5       2      1              1         6                       6
q6       1      1              1         5                       5
q7       1      2              1         8                       8
q8       1      1              1         5                       5

matches n_ctrl + 3 n_tgt + tau on every data qubit: True


## 2. Converting a native gate to a Stim gate

Each native gate is converted through its exact unitary (`stim.Tableau.from_unitary_matrix`),
so no angle is translated by hand. The test case is a single `cx(q, a)` compiled by Selene.

Stim does **not** reject a non-Clifford unitary: it silently returns some Clifford. The copy
therefore rebuilds the unitary from each converted tableau and requires it to equal the
original up to a global phase; the last lines show `Rz(0.3)` being refused.

In [3]:
TINY = """from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import output
from guppylang.std.quantum import cx, measure, qubit
from guppylang.std.qsystem import phased_x


@guppy
def prog() -> None:
    q = qubit()
    a = qubit()
    phased_x(q, angle(0.0), angle(0.0))
    cx(q, a)
    output("q", measure(q).read())
    output("a", measure(a).read())
"""
# Guppy reads the decorated function's source from disk, so the program is a real module file
_path = pathlib.Path(tempfile.mkdtemp(prefix='guppy_06a_')) / 'tiny_06a.py'
_path.write_text(TINY)
_spec = importlib.util.spec_from_file_location('tiny_06a', _path)
_mod = importlib.util.module_from_spec(_spec); sys.modules['tiny_06a'] = _mod
_spec.loader.exec_module(_mod)
TINY_INST = build(_mod.prog.compile())


def tiny_log(error_model):
    hook = CircuitExtractor()
    for shot in TINY_INST.run_shots(simulator=Stim(), n_qubits=2, n_shots=1, error_model=error_model,
                                    random_seed=1, event_hook=hook):
        result = list(shot)
    s = hook.shots[0]
    strip = lambda L: [o for o in L if o['op'] != 'BatchStart']
    return strip(s.get_error_model_input()), s.get_error_model_output(), result


ideal_in, ideal_out, _ = tiny_log(IdealErrorModel())
gates = [o for o in ideal_out if o['op'] in ('Rxy', 'Rz', 'Rzz')]
print("cx(q0, q1) as Selene compiles it (after the idle Rxy(0,0) on q0):")
for o in gates[1:]:
    print("  ", o)
c = stim.Circuit()
for o in gates[1:]:
    sm._append_remapped(c, *sm._clifford(o))
print("\nconverted:", str(c).replace('\n', ' | '))
print("equals CX(0, 1):", c.to_tableau() == stim.Tableau.from_named_gate('CX'))

silent = stim.Tableau.from_unitary_matrix(sm._rz(0.3), endian='little')
print(f"\nStim on Rz(0.3), unchecked: {silent.to_circuit()!s:s}   <- silently wrong")
try:
    sm._clifford({'op': 'Rz', 'qubit': 0, 'theta': 0.3})
except ValueError as e:
    print("selene_mirror on Rz(0.3):", e)

cx(q0, q1) as Selene compiles it (after the idle Rxy(0,0) on q0):
   {'op': 'Rxy', 'qubit': 1, 'theta': -1.5707963267948966, 'phi': 1.5707963267948966}
   {'op': 'Rzz', 'qubit0': 0, 'qubit1': 1, 'theta': 1.5707963267948966}
   {'op': 'Rz', 'qubit': 0, 'theta': -1.5707963267948966}
   {'op': 'Rxy', 'qubit': 1, 'theta': 1.5707963267948966, 'phi': 3.141592653589793}
   {'op': 'Rz', 'qubit': 1, 'theta': -1.5707963267948966}

converted: H 1 | S 1 1 0 | H 1 | CX 0 1 | H 1 | S 1 0 0 0 1 | H 1 | S 1 1 1 1
equals CX(0, 1): True

Stim on Rz(0.3), unchecked: H 0 0   <- silently wrong
selene_mirror on Rz(0.3): not a Clifford gate, cannot be mirrored: {'op': 'Rz', 'qubit': 0, 'theta': 0.3}


## 3. Where Selene puts its noise

`DepolarizingErrorModel` is a compiled library, so its placement is read off Selene's own
instruction log: the tiny program at `p_1q = 1` (an error on every single-qubit gate) and at
`p_meas = 1` (every readout flipped), comparing what the error model **receives** with what
it **emits**.

In [4]:
def is_pauli(o):
    return ((o['op'] == 'Rz' and math.isclose(abs(o['theta']), math.pi)) or
            (o['op'] == 'Rxy' and math.isclose(abs(o['theta']), math.pi)))


emin, emout, res1 = tiny_log(DepolarizingErrorModel(random_seed=3, p_1q=1.0))
print("p_1q = 1, error model output (inserted errors marked):")
inserted, before_ok = 0, True
for i, o in enumerate(emout):
    is_err = o not in ideal_out[i - inserted:i - inserted + 1] and is_pauli(o)
    print(f"   {'ERROR ->' if is_err else '        '} {o}")
    if is_err:
        nxt = emout[i + 1]
        before_ok &= nxt['op'] in ('Rxy', 'Rz') and nxt.get('qubit') == o['qubit']
        inserted += 1
n_1q = sum(o['op'] in ('Rxy', 'Rz') for o in ideal_out)
print(f"\n{inserted} errors inserted for {n_1q} native single-qubit gates; every one is a Pauli "
      f"immediately BEFORE a gate on the same qubit: {before_ok}")
print(f"errors inserted == native 1q gates ({inserted} == {n_1q}), so none on Rzz at p_2q = 0: {inserted == n_1q}")

_, mout, res_m = tiny_log(DepolarizingErrorModel(random_seed=3, p_meas=1.0))
print(f"\np_meas = 1: operations unchanged: {mout == ideal_out}; results {res_m} (noiseless: all 0)")
print("-> p_meas flips the reported bit only; no gate is applied to the qubit")

p_1q = 1, error model output (inserted errors marked):
            {'op': 'Reset', 'qubit': 0}
   ERROR -> {'op': 'Rz', 'qubit': 0, 'theta': 3.141592653589793}
            {'op': 'Rxy', 'qubit': 0, 'theta': 0.0, 'phi': 0.0}
            {'op': 'Reset', 'qubit': 1}
   ERROR -> {'op': 'Rxy', 'qubit': 1, 'theta': 3.141592653589793, 'phi': 0.0}
            {'op': 'Rxy', 'qubit': 1, 'theta': -1.5707963267948966, 'phi': 1.5707963267948966}
            {'op': 'Rzz', 'qubit0': 0, 'qubit1': 1, 'theta': 1.5707963267948966}
   ERROR -> {'op': 'Rxy', 'qubit': 0, 'theta': 3.141592653589793, 'phi': 0.0}
            {'op': 'Rz', 'qubit': 0, 'theta': -1.5707963267948966}
   ERROR -> {'op': 'Rxy', 'qubit': 1, 'theta': 3.141592653589793, 'phi': 1.5707963267948966}
            {'op': 'Rxy', 'qubit': 1, 'theta': 1.5707963267948966, 'phi': 3.141592653589793}
   ERROR -> {'op': 'Rz', 'qubit': 1, 'theta': 3.141592653589793}
            {'op': 'Rz', 'qubit': 1, 'theta': -1.5707963267948966}
            {'op': 

So the copy places `DEPOLARIZE1(p_1q)` before each native single-qubit gate, including the
idle `Rxy(0, 0)`, and `M(p_meas)` on each measurement; resets are noiseless (`p_init = 0`).
`p_2q` is zero throughout 06, and its placement has not been read off the log, so
`selene_mirror.to_stim` refuses `p_2q > 0` by default.

## 4. The noiseless copy

Every detector and the logical observable must be deterministic and zero, and Selene's own
noiseless shots must agree. Stim also refuses to build an error model if any detector is not
deterministic, so building one is a second check.

In [5]:
for tau, R in ((1, 3), (8, 12)):
    c0 = sm.mirror_circuit(G, tau, R, 0.0, 0.0)
    d0, o0 = c0.compile_detector_sampler().sample(2000, separate_observables=True)
    syn, dat = G.run(tau, R, 200, p_1q=0.0, p_meas=0.0)
    print(f"tau={tau:2d} R={R:2d}: {c0.num_detectors} detectors; copy: bits set {int(d0.sum())}, "
          f"observable {int(o0.sum())}; Selene: bits set {int(G.detectors(syn, dat).sum())}, "
          f"observable {int(G.observable(dat).sum())}")
    assert d0.sum() == 0 and o0.sum() == 0
dem = sm.mirror_circuit(G, 1, 3, 0.0005, 0.002).detector_error_model(decompose_errors=True)
print(f"\nerror model builds at Selene's noise: {dem.num_errors} fault mechanisms")

tau= 1 R= 3: 16 detectors; copy: bits set 0, observable 0; Selene: bits set 0, observable 0


tau= 8 R=12: 52 detectors; copy: bits set 0, observable 0; Selene: bits set 0, observable 0

error model builds at Selene's noise: 55 fault mechanisms


## 5. The noisy copy against Selene

At raised noise (`p_1q = 0.002`, `p_meas = 0.004`, Δt = 8, N = 12; 20,000 shots per block),
three statistics are compared:

- **per-detector firing rates** (36 detectors);
- **pairwise co-firing rates** `P(D_i and D_j)` for every pair with at least 20 expected
  co-firings: the correlation structure a decoder depends on;
- the **raw logical flip rate**, before any decoding.

Each is reported as χ²/dof of z-scores. Pairs share detectors and so are not independent,
which inflates χ² above 1 even for identical distributions; the reference is therefore what
two **independent blocks of Selene shots** give against each other (and two independent copy
samples). The copy passes if Selene-vs-copy sits within that spread.

In [6]:
CHK = dict(tau=8, rounds=12, shots=20_000, p_1q=0.002, p_meas=0.004)
cm = sm.mirror_circuit(G, CHK['tau'], CHK['rounds'], CHK['p_1q'], CHK['p_meas'])


def selene_block(offset):
    s, d = G.run(CHK['tau'], CHK['rounds'], CHK['shots'], offset=offset, seed=11,
                 p_1q=CHK['p_1q'], p_meas=CHK['p_meas'])
    return G.detectors(s, d).astype(bool), G.observable(d).astype(bool)


def copy_block(seed):
    d, o = cm.compile_detector_sampler(seed=seed).sample(CHK['shots'], separate_observables=True)
    return d, o[:, 0]


def compare_stats(A, B):
    (dA, oA), (dB, oB) = A, B
    n = len(dA)
    pA, pB = dA.mean(0), dB.mean(0)
    z = (pA - pB) / (np.sqrt((pA * (1 - pA) + pB * (1 - pB)) / n) + 1e-12)
    iu = np.triu_indices(dA.shape[1], 1)
    cA = (dA.T.astype(np.int32) @ dA.astype(np.int32))[iu] / n
    cB = (dB.T.astype(np.int32) @ dB.astype(np.int32))[iu] / n
    k = (cA + cB) * n / 2 >= 20
    zc = (cA - cB)[k] / np.sqrt((cA * (1 - cA) + cB * (1 - cB))[k] / n)
    a, b = oA.mean(), oB.mean()
    return {'per-detector chi2/dof': round(float(np.mean(z ** 2)), 2),
            'per-detector max |z|': round(float(np.abs(z).max()), 2),
            'pairwise chi2/dof': round(float(np.mean(zc ** 2)), 2),
            'pairs': int(k.sum()),
            'logical flip z': round(float((a - b) / np.sqrt((a * (1 - a) + b * (1 - b)) / n)), 2)}


S1, S2 = selene_block(0), selene_block(CHK['shots'])
M1, M2 = copy_block(21), copy_block(22)
stats = pd.DataFrame({'Selene vs Selene (reference)': compare_stats(S1, S2),
                      'copy vs copy (reference)': compare_stats(M1, M2),
                      'Selene vs copy, block 1': compare_stats(S1, M1),
                      'Selene vs copy, block 2': compare_stats(S2, M2)}).T
print(f"{CHK}\n")
print(stats.to_string())

{'tau': 8, 'rounds': 12, 'shots': 20000, 'p_1q': 0.002, 'p_meas': 0.004}

                              per-detector chi2/dof  per-detector max |z|  pairwise chi2/dof   pairs  logical flip z
Selene vs Selene (reference)                   0.90                  2.83               1.05  1324.0           -0.95
copy vs copy (reference)                       1.55                  3.08               1.08  1321.0            0.50
Selene vs copy, block 1                        1.33                  3.31               1.14  1321.0           -0.78
Selene vs copy, block 2                        1.07                  2.36               0.98  1323.0            0.67


## 6. What the copy's error model contains

Stim's error model for the circuit at Selene's noise (`p_1q = 0.0005`, `p_meas = 0.002`),
N = 8, for three idle intervals. Each fault mechanism is classed by the shape of what it lights
up, and the table gives each class's share of total fault probability. A hand-built
phenomenological graph has space-like, time-like and boundary edges, but **no diagonal ones**:
a fault part-way through extraction lights one check in one round and a different check in
the next.

In [7]:
def anatomy(tau, R=8, p_1q=0.0005, p_meas=0.002, geom=None):
    g = geom or G
    nz = len(g.zcols)
    dem = sm.mirror_circuit(g, tau, R, p_1q, p_meas).detector_error_model(decompose_errors=True)
    mass = collections.Counter()
    for inst in dem.flattened():
        if inst.type != 'error':
            continue
        p = inst.args_copy()[0]
        dets = [t.val for t in inst.targets_copy() if t.is_relative_detector_id()]
        if any(t.is_separator() for t in inst.targets_copy()) or len(dets) > 2:
            kind = 'more than 2 detectors'
        elif len(dets) == 1:
            kind = 'boundary (1 detector)'
        else:
            (r1, c1), (r2, c2) = [(d // nz, d % nz) for d in dets]
            kind = ('space-like (same round)' if r1 == r2 else
                    'time-like (same check)' if c1 == c2 else 'DIAGONAL (other check, other round)')
        mass[kind] += p
    tot = sum(mass.values())
    return {k: round(100 * v / tot, 1) for k, v in mass.items()}


ANAT = pd.DataFrame({f'tau={t}': anatomy(t) for t in (1, 8, 32)}).fillna(0.0)
ext = {f'tau={t}': round(100 * G.ops_mean / (G.ops_mean + t), 1) for t in (1, 8, 32)}
print("% of total fault probability, by symptom shape (R = 8)\n")
print(ANAT.to_string())
print("\nextraction share of a data qubit's charged ops per round, n_data / (n_data + tau) (%):", ext)

% of total fault probability, by symptom shape (R = 8)

                                     tau=1  tau=8  tau=32
space-like (same round)               12.4   19.2    26.7
time-like (same check)                45.5   30.8    14.8
DIAGONAL (other check, other round)    8.4    5.7     2.7
boundary (1 detector)                 33.8   44.3    55.8

extraction share of a data qubit's charged ops per round, n_data / (n_data + tau) (%): {'tau=1': 87.3, 'tau=8': 46.3, 'tau=32': 17.7}


## 6b. The copy at d = 5

Before any shots are generated at a new distance, `scripts/circuit_level_generate.py
--verify-only` runs §4's noiseless check and §5's statistics there and writes them to
`results/circuit_level/verify_mirror_<code>_d<d>.json`; generation does not start unless they
pass (noiseless exactly zero, and Selene-vs-copy χ²/dof within 1.5× of the larger reference).
The table reads those files back. The d = 3 row reproduces §5 exactly, so the two are the
same check. Then the error-model content at d = 5, for the Δt range of its sweep.

In [8]:
import json
rows = []
for d in (3, 5):
    r = json.loads((pathlib.Path('..') / 'results' / 'circuit_level' / f'verify_mirror_{CODE}_d{d}.json').read_text())
    st = r['statistics']
    rows.append({'d': d, 'pass': r['pass'],
                 'noiseless bits set': sum(v['bits'] + v['observable'] for k, v in r.items() if k.startswith('noiseless')),
                 'per-det: references': f"{st['Selene vs Selene']['per-detector chi2/dof']:.2f} / {st['copy vs copy']['per-detector chi2/dof']:.2f}",
                 'per-det: Selene vs copy': f"{st['Selene vs copy 1']['per-detector chi2/dof']:.2f} / {st['Selene vs copy 2']['per-detector chi2/dof']:.2f}",
                 'pairwise: references': f"{st['Selene vs Selene']['pairwise chi2/dof']:.2f} / {st['copy vs copy']['pairwise chi2/dof']:.2f}",
                 'pairwise: Selene vs copy': f"{st['Selene vs copy 1']['pairwise chi2/dof']:.2f} / {st['Selene vs copy 2']['pairwise chi2/dof']:.2f}",
                 'pairs': st['Selene vs copy 1']['pairs']})
print(pd.DataFrame(rows).set_index('d').to_string())

G5 = Geometry(CODE, 5)
ANAT5 = pd.DataFrame({f'tau={t}': anatomy(t, geom=G5) for t in (1, 4, 16)}).fillna(0.0)
print(f"\nd=5: % of total fault probability, by symptom shape (R = 8)\n")
print(ANAT5.to_string())
print("\nextraction share at d=5 (%):", {f'tau={t}': round(100 * G5.ops_mean / (G5.ops_mean + t), 1) for t in (1, 4, 16)})

   pass  noiseless bits set per-det: references per-det: Selene vs copy pairwise: references pairwise: Selene vs copy  pairs
d                                                                                                                           
3  True                   0         0.90 / 1.55             1.33 / 1.07          1.05 / 1.08              1.14 / 0.98   1321
5  True                   0         1.10 / 0.89             0.89 / 1.07          1.01 / 0.99              0.98 / 0.99  12074



d=5: % of total fault probability, by symptom shape (R = 8)

                                     tau=1  tau=4  tau=16
space-like (same round)               21.1   27.0    39.4
time-like (same check)                45.3   38.5    24.1
DIAGONAL (other check, other round)   14.0   11.9     7.4
boundary (1 detector)                 19.7   22.7    29.1

extraction share at d=5 (%): {'tau=1': 89.4, 'tau=4': 67.7, 'tau=16': 34.4}


## 7. Findings

**1. The copy runs the same native gates as Selene.** One extraction round compiles to one
`Rzz` per CNOT (24) and, on each data qubit, one single-qubit gate per CNOT it controls, three
per CNOT it targets, plus Δt idle `Rxy(0, 0)`: `n_ctrl + 3·n_tgt + Δt` holds on all 9 data qubits.
The control/target asymmetry is the one 04 measured through noise (1.4 against 3.9 charged
ops per CNOT end).

**2. Every gate converts exactly, and a wrong conversion cannot pass silently.** Selene's CNOT
decomposition converts to exactly `CX`. Stim's `from_unitary_matrix` returns a wrong Clifford
for a non-Clifford gate without complaint (`Rz(0.3)` becomes the identity), so every
conversion is checked against the original unitary and a mismatch raises.

**3. Selene puts one random Pauli before every native single-qubit gate, and nothing else.**
Read directly off its instruction log at `p_1q = 1`: 5 errors for 5 single-qubit gates, each
immediately before its gate on the same qubit, including the idle `Rxy(0, 0)`; no error on
`Rzz` at `p_2q = 0`. At `p_meas = 1` the operations are unchanged and the results flip: a
classical record flip. (For single-qubit depolarizing noise, before and after a Clifford gate
are the same channel, so the placement matters only for consistency; it is matched anyway.)

**4. The noiseless copy is deterministic**, as is Selene: zero detection events and a zero
observable at both settings tested, and Stim builds the error model (55 mechanisms at Δt = 1,
N = 3), which it refuses to do for any non-deterministic detector.

**5. With noise, the copy cannot be told apart from Selene at 20,000 shots.** Per-detector
χ²/dof is 1.07 and 1.33 for the two Selene blocks against the copy, inside the 0.90–1.55 that
the two references give. Pairwise co-firing gives 0.98 and 1.14 against references of 1.05
and 1.08; the 0.16 spread between the two Selene-vs-copy blocks shows how much this statistic
varies between samples, so 1.14 is within it. The logical flip rate agrees at z = −0.78 and
0.67.

**6. The error model has mechanisms the phenomenological graph lacks, and fewer of them at
long intervals.** Diagonal mechanisms carry 8.4%, 5.7% and 2.7% of fault probability at
Δt = 1, 8 and 32, as the extraction share of a data qubit's noise falls from 87% to 46% to 18%.
Time-like mechanisms fall from 45.5% to 14.8% over the same range.

**7. The copy holds at d = 5 too, and its diagonal mechanisms matter more there.** The same
checks at d = 5 pass: noiseless detectors exactly zero, per-detector χ²/dof 0.89 and 1.07 against
references of 1.10 and 0.89, pairwise 0.98 and 0.99 against 1.01 and 0.99 over 12,074 detector
pairs. Diagonal mechanisms carry 14.0%, 11.9% and 7.4% of fault probability at Δt = 1, 4 and 16,
about 1.7× their d = 3 share at similar extraction shares.

### Limits

- **Two distances** (rotated d = 3 and 5) and `p_2q = 0`. The copy itself is general, but the
  checks have been run at these two only; the two-qubit noise placement has not been read off
  the log.
- **20,000 shots per statistical comparison.** A mismatch smaller than about 1% in a detector
  rate would not be resolved.
- **Same package versions as the original runs** (selene-sim 0.3.0, compiler 0.4.2). A compiler
  change could change the native gate sequence; the copy reads it fresh each time, but these
  checks should be re-run.